# Monotonic SOH constraint (Week 6)

Predict **EOL** (end-of-life) from the first **100 cycles**, same as Week 5 — but also predict **SOH** (state of health) at each cycle so we can check whether the health curve makes physical sense.

**Real batteries:** health usually stays flat or **decreases** over time. It should not **increase** cycle-to-cycle.

| Step | What |
|------|------|
| **A** *(this notebook section)* | Load Week 5 data; define dual-head GRU skeleton |
| B | Train **unconstrained** model (no SOH penalty) |
| C | Train **constrained** model (penalty when predicted SOH goes up) |
| D | EOL metrics + SOH trajectory figures |

Same split as Weeks 4–5: `cell_split.csv` (94 train / 20 val / 20 test).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'data' / 'raw').is_dir():
            return candidate
    raise FileNotFoundError(f'Could not find data/raw/ starting from {here}')


ROOT = find_repo_root()
TARGETS_PATH = ROOT / 'data' / 'cell_targets.csv'
SUMMARY_PATH = ROOT / 'data' / 'processed' / 'cycle_summary.csv'
SPLIT_PATH = ROOT / 'data' / 'processed' / 'cell_split.csv'

SEQ_LEN = 100
CYCLE_MIN = 1  # exclude formation cycle 0
CYCLE_MAX = SEQ_LEN
SEQUENCE_COLS = (
    'soh',
    'dc_internal_resistance',
    'energy_efficiency',
    'temperature_average',
)
TARGET = 'EOL'
SOH_CHANNEL = 0  # first channel in the tensor is true SOH

print('Project root:', ROOT)
print('Sequence length:', SEQ_LEN, 'cycles')
print('Channels:', ', '.join(SEQUENCE_COLS))

## Step A — load data (same as Week 5)

Each cell becomes one tensor of shape **(100 cycles, 4 channels)**:

1. **SOH** — discharge capacity ÷ initial capacity
2. Resistance, energy efficiency, temperature (from `cycle_summary.csv`)

Target label: **EOL** (first cycle where capacity drops below 80% of initial).

In [ ]:
targets = pd.read_csv(TARGETS_PATH)
split_df = pd.read_csv(SPLIT_PATH)

labels = targets.merge(split_df, on=['file_id', 'cell_id'], validate='one_to_one')

assert len(targets) == 134
assert len(labels) == 134
assert labels['file_id'].is_unique
assert set(labels['split']) == {'train', 'val', 'test'}

print(f'Cells: {len(labels)}')
print(labels['split'].value_counts().sort_index().to_string())
print(f'EOL range: {labels[TARGET].min()} – {labels[TARGET].max()} cycles')

In [ ]:
summary_cols = [
    'file_id',
    'cell_id',
    'cycle_index',
    'discharge_capacity',
    'dc_internal_resistance',
    'energy_efficiency',
    'temperature_average',
]
cycle_summary = pd.read_csv(SUMMARY_PATH, usecols=summary_cols)
cycle_summary = cycle_summary[
    (cycle_summary['cycle_index'] >= CYCLE_MIN)
    & (cycle_summary['cycle_index'] <= CYCLE_MAX)
].copy()

print(f'cycle_summary rows (cycles {CYCLE_MIN}–{CYCLE_MAX}): {len(cycle_summary):,}')
print(f'Unique cells in summary: {cycle_summary["file_id"].nunique()}')

In [ ]:
def build_sequence(group: pd.DataFrame, initial_capacity: float) -> np.ndarray:
    """Return array shape (SEQ_LEN, n_channels) for one cell."""
    g = group.sort_values('cycle_index')
    expected_cycles = np.arange(CYCLE_MIN, CYCLE_MAX + 1)
    if not np.array_equal(g['cycle_index'].to_numpy(), expected_cycles):
        missing = set(expected_cycles) - set(g['cycle_index'])
        raise ValueError(f'Missing cycles {sorted(missing)[:5]}... (need {SEQ_LEN} consecutive cycles)')

    soh = g['discharge_capacity'].to_numpy(dtype=float) / initial_capacity
    out = np.column_stack([
        soh,
        g['dc_internal_resistance'].to_numpy(dtype=float),
        g['energy_efficiency'].to_numpy(dtype=float),
        g['temperature_average'].to_numpy(dtype=float),
    ])
    return out


sequences = []
y = []
meta_rows = []

for row in labels.itertuples(index=False):
    group = cycle_summary[cycle_summary['file_id'] == row.file_id]
    seq = build_sequence(group, row.initial_capacity)
    sequences.append(seq)
    y.append(row.EOL)
    meta_rows.append({
        'file_id': row.file_id,
        'cell_id': row.cell_id,
        'split': row.split,
        TARGET: row.EOL,
        'initial_capacity': row.initial_capacity,
    })

X = np.stack(sequences, axis=0)
y = np.array(y, dtype=float)
meta = pd.DataFrame(meta_rows)

# True SOH trajectory (unscaled) — used later to train/plot the SOH head
soh_true = X[:, :, SOH_CHANNEL].copy()

print('X shape:', X.shape, '  (cells, cycles, channels)')
print('soh_true shape:', soh_true.shape, '  (cells, cycles)')
print('y shape:', y.shape)

In [ ]:
assert X.shape == (134, SEQ_LEN, len(SEQUENCE_COLS))
assert soh_true.shape == (134, SEQ_LEN)
assert len(y) == 134
assert not np.isnan(X).any(), 'NaN in sequence tensor — check cycle_summary'
assert (soh_true > 0).all(), 'SOH should be positive'

for split_name in ('train', 'val', 'test'):
    n = (meta['split'] == split_name).sum()
    assert n in (94, 20), f'unexpected {split_name} count: {n}'

print('Checks passed.')
print(f'SOH range (all cells, cycles 1–{SEQ_LEN}): {soh_true.min():.4f} – {soh_true.max():.4f}')
print('Note: SOH can exceed 1.0 — initial capacity is taken at cycle 1, so a later cycle can read slightly higher.')
print()
print('Example — first train cell, cycle 1 and cycle 100:')
train_idx = meta.index[meta['split'] == 'train'][0]
print(meta.loc[train_idx, ['file_id', TARGET, 'split']].to_string())
print('cycle 1 SOH:', round(soh_true[train_idx, 0], 4))
print(f'cycle {SEQ_LEN} SOH:', round(soh_true[train_idx, -1], 4))

## Step A — scale inputs and EOL target

Same preprocessing as Week 5:

1. **Input channels** — `StandardScaler` fit on train timesteps only.
2. **EOL target** — standardize using train mean/std (helps the GRU learn).
3. **True SOH** — kept in raw 0–1 scale (not scaled) for the SOH head and plots.

In [ ]:
import torch
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
N_CHANNELS = len(SEQUENCE_COLS)

# Week 5 best hyperparameters — fixed for Week 6 (we tune penalty weight later)
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2
LEARNING_RATE = 3e-4
BATCH_SIZE = 16
MAX_EPOCHS = 200
PATIENCE = 20

torch.manual_seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

train_mask = meta['split'].eq('train').to_numpy()
val_mask = meta['split'].eq('val').to_numpy()
test_mask = meta['split'].eq('test').to_numpy()

scaler_x = StandardScaler()
n_cells, seq_len, n_feat = X.shape
scaler_x.fit(X[train_mask].reshape(-1, n_feat))
X_scaled = scaler_x.transform(X.reshape(-1, n_feat)).reshape(X.shape)

y_mean = y[train_mask].mean()
y_std = y[train_mask].std()
y_scaled = (y - y_mean) / y_std

X_train = X_scaled[train_mask]
X_val = X_scaled[val_mask]
X_test = X_scaled[test_mask]
y_train = y_scaled[train_mask]
y_val = y_scaled[val_mask]
y_test = y_scaled[test_mask]
y_train_raw = y[train_mask]
y_val_raw = y[val_mask]
y_test_raw = y[test_mask]

soh_train = soh_true[train_mask]
soh_val = soh_true[val_mask]
soh_test = soh_true[test_mask]

print(f'train {len(y_train)} | val {len(y_val)} | test {len(y_test)}')
print(f'EOL train mean {y_mean:.0f}, std {y_std:.0f}')

## Step A — dual-head GRU skeleton

Week 5 GRU had **one output**: predicted EOL.

Week 6 adds a **second output**: predicted SOH at **every cycle**.

```
Input (100 cycles × 4 channels)
        ↓
      GRU  → hidden state at each cycle
        ↓
   ┌────┴────┐
   ↓         ↓
 EOL head   SOH head (one value per cycle)
 (1 number) (100 numbers)
```

Training comes in Steps B and C. Here we only define the model and check output shapes.

In [ ]:
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def unscale_predictions(pred_scaled: np.ndarray) -> np.ndarray:
    return pred_scaled * y_std + y_mean


class GRUDualHead(nn.Module):
    """GRU with two outputs: EOL (scalar) and SOH trajectory (per cycle)."""

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        num_layers: int,
        dropout: float,
    ) -> None:
        super().__init__()
        gru_dropout = dropout if num_layers > 1 else 0.0
        self.gru = nn.GRU(
            input_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=gru_dropout,
        )
        self.dropout = nn.Dropout(dropout)
        self.eol_head = nn.Linear(hidden_size, 1)
        self.soh_head = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Return (eol_pred_scaled, soh_pred) with shapes (batch,) and (batch, seq_len)."""
        hidden, _ = self.gru(x)  # (batch, seq_len, hidden_size)
        eol = self.eol_head(self.dropout(hidden[:, -1, :])).squeeze(-1)
        soh = self.soh_head(hidden).squeeze(-1)  # (batch, seq_len)
        return eol, soh


def monotonic_violation_count(soh_pred: np.ndarray) -> int:
    """Count cycle pairs where predicted SOH increases (for evaluation later)."""
    increases = soh_pred[:, 1:] > soh_pred[:, :-1]
    return int(increases.sum())


def make_loader(
    x_arr: np.ndarray,
    y_arr: np.ndarray,
    soh_arr: np.ndarray,
    shuffle: bool,
) -> DataLoader:
    ds = TensorDataset(
        torch.tensor(x_arr, dtype=torch.float32),
        torch.tensor(y_arr, dtype=torch.float32),
        torch.tensor(soh_arr, dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

In [ ]:
# Quick shape check — random weights, no training yet
model = GRUDualHead(N_CHANNELS, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(device)
sample_x = torch.tensor(X_train[:4], dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    eol_pred, soh_pred = model(sample_x)

print('Sample batch size:', sample_x.shape[0])
print('EOL pred shape:', tuple(eol_pred.shape), '  (one number per cell)')
print('SOH pred shape:', tuple(soh_pred.shape), '  (one number per cell per cycle)')
print()
print('Example SOH curve (cell 0, first 5 cycles):', soh_pred[0, :5].cpu().numpy().round(4))
print('True SOH (cell 0, first 5 cycles):           ', soh_train[0, :5].round(4))

## Step B — train unconstrained model

**Unconstrained** = no monotonic penalty. The model learns two things at once:

1. **EOL loss** — how close is the predicted end-of-life? (main task, same as Week 5)
2. **SOH loss** — how close is the predicted health curve to the true curve? (teaches the SOH head to draw a real shape)

Total loss = EOL loss + **α** × SOH loss, with **α = 0.1** (SOH is a helper, not the main goal).

Early stopping uses **validation EOL MAE** (mean absolute error), same as Week 5.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
    return {'mae': mae, 'rmse': rmse, 'mape': mape}


def predict_eol_cycles(model: nn.Module, x_arr: np.ndarray) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        eol_scaled, _ = model(torch.tensor(x_arr, dtype=torch.float32).to(device))
    return unscale_predictions(eol_scaled.cpu().numpy())


def predict_soh(model: nn.Module, x_arr: np.ndarray) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        _, soh_pred = model(torch.tensor(x_arr, dtype=torch.float32).to(device))
    return soh_pred.cpu().numpy()


def monotonic_penalty(soh_pred: torch.Tensor) -> torch.Tensor:
    """Average amount predicted SOH increases from one cycle to the next."""
    increases = torch.relu(soh_pred[:, 1:] - soh_pred[:, :-1])
    return increases.mean()


def train_dual_gru(
    soh_alpha: float = 0.1,
    monotonic_lambda: float = 0.0,
    verbose: bool = False,
) -> tuple[GRUDualHead, float, dict[str, torch.Tensor]]:
    """Train dual-head GRU. monotonic_lambda=0 → unconstrained (Step B)."""
    torch.manual_seed(RANDOM_STATE)
    model = GRUDualHead(N_CHANNELS, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    eol_loss_fn = nn.MSELoss()
    soh_loss_fn = nn.MSELoss()
    train_loader = make_loader(X_train, y_train, soh_train, shuffle=True)

    best_val_mae = float('inf')
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for xb, yb, soh_b in train_loader:
            xb, yb, soh_b = xb.to(device), yb.to(device), soh_b.to(device)
            optimizer.zero_grad()
            eol_pred, soh_pred = model(xb)
            loss = eol_loss_fn(eol_pred, yb) + soh_alpha * soh_loss_fn(soh_pred, soh_b)
            if monotonic_lambda > 0:
                loss = loss + monotonic_lambda * monotonic_penalty(soh_pred)
            loss.backward()
            optimizer.step()

        val_mae = float(np.abs(predict_eol_cycles(model, X_val) - y_val_raw).mean())
        if val_mae < best_val_mae - 1e-4:
            best_val_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if verbose and (epoch == 1 or epoch % 20 == 0):
            train_mae = float(np.abs(predict_eol_cycles(model, X_train) - y_train_raw).mean())
            print(f'  epoch {epoch:3d}  train EOL MAE {train_mae:6.1f}  val EOL MAE {val_mae:6.1f}')

        if epochs_no_improve >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return model, best_val_mae, best_state

In [ ]:
SOH_ALPHA = 0.1  # weight on SOH reconstruction loss

unconstrained_model, unconstrained_val_mae, unconstrained_state = train_dual_gru(
    soh_alpha=SOH_ALPHA,
    monotonic_lambda=0.0,
    verbose=True,
)
print(f'\nUnconstrained — best val EOL MAE: {unconstrained_val_mae:.1f} cycles')
print('(Week 5 single-head GRU test MAE was about 111 cycles)')

In [ ]:
uc_eol_train = predict_eol_cycles(unconstrained_model, X_train)
uc_eol_val = predict_eol_cycles(unconstrained_model, X_val)
uc_eol_test = predict_eol_cycles(unconstrained_model, X_test)

uc_soh_train = predict_soh(unconstrained_model, X_train)
uc_soh_val = predict_soh(unconstrained_model, X_val)
uc_soh_test = predict_soh(unconstrained_model, X_test)

unconstrained_metrics = {
    'train': regression_metrics(y_train_raw, uc_eol_train),
    'val': regression_metrics(y_val_raw, uc_eol_val),
    'test': regression_metrics(y_test_raw, uc_eol_test),
}

pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'MAE (cycles)': [unconstrained_metrics[s]['mae'] for s in ('train', 'val', 'test')],
    'RMSE': [unconstrained_metrics[s]['rmse'] for s in ('train', 'val', 'test')],
    'MAPE (%)': [unconstrained_metrics[s]['mape'] for s in ('train', 'val', 'test')],
}).round(2)

In [ ]:
def violation_summary(soh_pred: np.ndarray, split_name: str) -> dict[str, float]:
    n_pairs = soh_pred.shape[0] * (soh_pred.shape[1] - 1)
    n_violations = monotonic_violation_count(soh_pred)
    return {
        'split': split_name,
        'violations': n_violations,
        'violation_rate (%)': round(100 * n_violations / n_pairs, 2),
    }


pd.DataFrame([
    violation_summary(uc_soh_train, 'train'),
    violation_summary(uc_soh_val, 'val'),
    violation_summary(uc_soh_test, 'test'),
])

**Step B complete.** The unconstrained model is saved in memory as `unconstrained_model`.

Next (Step C): train the **constrained** model — same setup, but add a penalty when predicted SOH goes up cycle-to-cycle.

## Step C — train constrained model

**Constrained** = same as Step B, plus a **monotonic penalty** when predicted SOH goes **up** from one cycle to the next.

We try a few penalty strengths (**λ** = 0.01, 0.1, 1.0) and pick the one with the **lowest validation EOL MAE** — same selection rule as Week 4/5 hyperparameter tuning.

In [ ]:
LAMBDA_GRID = (0.01, 0.1, 1.0)

lambda_results = []
best_constrained_model = None
best_constrained_state = None
best_lambda = None
best_constrained_val_mae = float('inf')

for lam in LAMBDA_GRID:
    print(f'Training with monotonic_lambda={lam} ...')
    model, val_mae, state = train_dual_gru(
        soh_alpha=SOH_ALPHA,
        monotonic_lambda=lam,
        verbose=False,
    )
    soh_val = predict_soh(model, X_val)
    val_violations = monotonic_violation_count(soh_val)
    row = {
        'monotonic_lambda': lam,
        'val_eol_mae': round(val_mae, 2),
        'val_violations': val_violations,
        'val_violation_rate (%)': round(100 * val_violations / (len(soh_val) * (SEQ_LEN - 1)), 2),
    }
    lambda_results.append(row)
    print(f'  val EOL MAE {val_mae:.1f}  |  val SOH violations {val_violations} ({row["val_violation_rate (%)"]}%)')

    if val_mae < best_constrained_val_mae:
        best_constrained_val_mae = val_mae
        best_lambda = lam
        best_constrained_model = model
        best_constrained_state = state

print(f'\nBest lambda: {best_lambda}  (val EOL MAE {best_constrained_val_mae:.1f} cycles)')
pd.DataFrame(lambda_results)

In [ ]:
constrained_model = best_constrained_model
constrained_model.load_state_dict(best_constrained_state)

c_eol_test = predict_eol_cycles(constrained_model, X_test)
c_soh_test = predict_soh(constrained_model, X_test)

constrained_metrics = {
    'train': regression_metrics(y_train_raw, predict_eol_cycles(constrained_model, X_train)),
    'val': regression_metrics(y_val_raw, predict_eol_cycles(constrained_model, X_val)),
    'test': regression_metrics(y_test_raw, c_eol_test),
}

comparison = pd.DataFrame([
    {
        'model': 'unconstrained',
        'monotonic_lambda': 0.0,
        'test_mae': round(unconstrained_metrics['test']['mae'], 2),
        'test_violation_rate (%)': violation_summary(uc_soh_test, 'test')['violation_rate (%)'],
    },
    {
        'model': 'constrained',
        'monotonic_lambda': best_lambda,
        'test_mae': round(constrained_metrics['test']['mae'], 2),
        'test_violation_rate (%)': violation_summary(c_soh_test, 'test')['violation_rate (%)'],
    },
])
comparison

In [ ]:
pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'unconstrained MAE': [unconstrained_metrics[s]['mae'] for s in ('train', 'val', 'test')],
    'constrained MAE': [constrained_metrics[s]['mae'] for s in ('train', 'val', 'test')],
}).round(2)

**Step C complete.** `constrained_model` uses the best **λ** from the grid above.

Next (Step D): save metrics JSON, plot example SOH curves (true vs unconstrained vs constrained), and model comparison figure.

## Step D — save metrics and figures

1. Write metrics JSON for unconstrained and constrained models
2. Plot **example SOH curves** on the test set (true vs both models)
3. Bar chart comparing **test EOL MAE** — XGBoost baseline, unconstrained GRU, constrained GRU

Example cells are auto-picked: the four test batteries with the most SOH violations in the unconstrained model.

In [ ]:
import json

import matplotlib.pyplot as plt

UC_METRICS_PATH = ROOT / 'results' / 'metrics' / 'gru_unconstrained.json'
C_METRICS_PATH = ROOT / 'results' / 'metrics' / 'gru_monotonic.json'
SOH_FIGURE_PATH = ROOT / 'results' / 'figures' / 'soh_curves_constrained.png'
COMPARE_FIGURE_PATH = ROOT / 'results' / 'figures' / 'model_comparison_soh_penalty.png'
XGB_METRICS_PATH = ROOT / 'results' / 'metrics' / 'xgboost.json'


def build_metrics_payload(
    model_name: str,
    metrics: dict[str, dict[str, float]],
    soh_preds: dict[str, np.ndarray],
    monotonic_lambda: float,
) -> dict:
    return {
        'model': model_name,
        'sequence_len': SEQ_LEN,
        'channels': list(SEQUENCE_COLS),
        'soh_alpha': SOH_ALPHA,
        'monotonic_lambda': monotonic_lambda,
        'best_params': {
            'hidden_size': HIDDEN_SIZE,
            'num_layers': NUM_LAYERS,
            'dropout': DROPOUT,
            'learning_rate': LEARNING_RATE,
        },
        'split': {
            'train': len(y_train),
            'val': len(y_val),
            'test': len(y_test),
            'random_state': RANDOM_STATE,
        },
        'train': metrics['train'],
        'val': metrics['val'],
        'test': metrics['test'],
        'soh_violations': {
            split_name: violation_summary(soh_preds[split_name], split_name)
            for split_name in ('train', 'val', 'test')
        },
    }


uc_payload = build_metrics_payload(
    'gru_dual_unconstrained',
    unconstrained_metrics,
    {'train': uc_soh_train, 'val': uc_soh_val, 'test': uc_soh_test},
    monotonic_lambda=0.0,
)
c_payload = build_metrics_payload(
    'gru_dual_monotonic',
    constrained_metrics,
    {
        'train': predict_soh(constrained_model, X_train),
        'val': predict_soh(constrained_model, X_val),
        'test': c_soh_test,
    },
    monotonic_lambda=best_lambda,
)
c_payload['lambda_grid_results'] = lambda_results

UC_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
UC_METRICS_PATH.write_text(json.dumps(uc_payload, indent=2))
C_METRICS_PATH.write_text(json.dumps(c_payload, indent=2))

print('Saved', UC_METRICS_PATH)
print('Saved', C_METRICS_PATH)
print(f"Unconstrained test MAE: {uc_payload['test']['mae']:.1f} cycles")
print(f"Constrained test MAE:   {c_payload['test']['mae']:.1f} cycles")

In [ ]:
def violations_per_cell(soh_pred: np.ndarray) -> np.ndarray:
    return (soh_pred[:, 1:] > soh_pred[:, :-1]).sum(axis=1)


test_meta = meta.loc[test_mask].reset_index(drop=True)
test_violations = violations_per_cell(uc_soh_test)
example_local_idx = np.argsort(test_violations)[-4:][::-1]

cycles = np.arange(CYCLE_MIN, CYCLE_MAX + 1)
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)
axes = axes.ravel()

for ax, local_i in zip(axes, example_local_idx):
    true_curve = soh_test[local_i]
    uc_curve = uc_soh_test[local_i]
    c_curve = c_soh_test[local_i]
    row = test_meta.iloc[local_i]

    ax.plot(cycles, true_curve, 'k-', linewidth=1.5, label='True SOH')
    ax.plot(cycles, uc_curve, '--', color='C1', linewidth=1.2, label='Unconstrained')
    ax.plot(cycles, c_curve, '-.', color='C2', linewidth=1.2, label='Constrained')
    ax.set_title(
        f"{row['cell_id']}  (EOL {row['EOL']:.0f}, "
        f"{int(test_violations[local_i])} uc violations)"
    )
    ax.set_xlim(CYCLE_MIN, CYCLE_MAX)

for ax in axes[2:]:
    ax.set_xlabel('Cycle index')
for ax in axes[::2]:
    ax.set_ylabel('SOH')
axes[0].legend(loc='lower left', fontsize=8)
fig.suptitle('Example test-set SOH trajectories (cycles 1–100)', y=1.02)
fig.tight_layout()
SOH_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(SOH_FIGURE_PATH, dpi=150, bbox_inches='tight')
plt.show()
print('Saved figure to', SOH_FIGURE_PATH)

In [ ]:
xgb_test_mae = json.loads(XGB_METRICS_PATH.read_text())['test']['mae']

compare_df = pd.DataFrame({
    'model': ['XGBoost (Week 4)', 'GRU unconstrained', 'GRU constrained'],
    'test_mae': [
        xgb_test_mae,
        unconstrained_metrics['test']['mae'],
        constrained_metrics['test']['mae'],
    ],
})

fig, ax = plt.subplots(figsize=(7, 4))
compare_df.set_index('model')['test_mae'].plot.bar(ax=ax, color=['C3', 'C1', 'C2'])
ax.set_ylabel('Test MAE (cycles)')
ax.set_title('EOL prediction — test set (n=20 cells)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
fig.tight_layout()
fig.savefig(COMPARE_FIGURE_PATH, dpi=150)
plt.show()
print('Saved figure to', COMPARE_FIGURE_PATH)
compare_df.round(2)

**Step D complete.** Week 6 notebook done.

| Output | Path |
|--------|------|
| Unconstrained metrics | `results/metrics/gru_unconstrained.json` |
| Constrained metrics | `results/metrics/gru_monotonic.json` |
| SOH curve examples | `results/figures/soh_curves_constrained.png` |
| Model comparison | `results/figures/model_comparison_soh_penalty.png` |

Next: `docs/week06/README.md`, slides, and report §5.3 / §6.3 (when you're ready).